# 🌍 Project: Multimodal AI Travel Agent
### Chat + Tools + Image Generation + Voice + PDF Export — in Gradio



**APIs used (all free, no signup):**
- Open-Meteo → live weather
- Frankfurter → live currency rates
- Your existing OpenAI key → chat, images, TTS

---

## Step 0 — Setup & Imports

In [5]:
#python
%pip install fpdf2


  Using cached fpdf2-2.8.7-py3-none-any.whl.metadata (81 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
Using cached fpdf2-2.8.7-py3-none-any.whl (327 kB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ataka\projects\llm_engineering\.venv\Scripts\python.exe -m pip install --upgrade pip


In [6]:
#python
import os
import base64        
import json          
import requests         

from io import BytesIO      
from PIL import Image       
from dotenv import load_dotenv    
from openai import OpenAI  

import gradio as gr    
from fpdf import FPDF      

load_dotenv(override=True)                       
openai = OpenAI()            
MODEL = "gpt-4.1-mini"      




## Step 1 — The System Prompt

In [9]:
#python
system_message = """You are TravelBuddy, a friendly expert travel agent for trips anywhere in the world."""

## Step 2 — Tool #1: Live Weather (Open-Meteo)
Two HTTP calls: one to turn a city name into coordinates, one to get the weather.

In [12]:
#python
def get_weather(city):
    # 1. Geocode the city name → latitude/longitude
    geo = requests.get(                      
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1}
    ).json()
    if "results" not in geo:
        return {"error": f"Could not find {city}"}
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    # 2. Get current weather for those coordinates
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current": "temperature_2m,weather_code,wind_speed_10m"}
    ).json()

    current = weather["current"]
    return {
        "city": city,
        "temperature_c": current["temperature_2m"],
        "wind_kmh": current["wind_speed_10m"],
    }
get_weather("Istanbul")

{'city': 'Istanbul', 'temperature_c': 21.6, 'wind_kmh': 21.6}

## Step 3 — Tool #2: Currency Conversion (Frankfurter)

In [14]:
#python
def convert_currency(amount, from_currency, to_currency):
    response = requests.get(
        f"https://api.frankfurter.app/latest",
        params={"amount": amount, "from": from_currency, "to": to_currency}
    )
    data = response.json()               
    if "rates" not in data:
        return {"error": "Conversion failed — check the currency codes"}
    return {
        "amount": amount,
        "from": from_currency,
        "converted": data["rates"][to_currency],
        "to": to_currency,
    }
convert_currency(100, "USD", "TRY")

{'amount': 100, 'from': 'USD', 'converted': 4626.09, 'to': 'TRY'}

## Step 4 — Tool #3: The Itinerary (App State!)

In [16]:
#python
itinerary = []   # each entry: {"day": 1, "activity": "Visit Hagia Sophia"}

def add_to_itinerary(day, activity):
    itinerary.append({"day": day, "activity": activity})
    return {"status": "added", "total_items": len(itinerary)}

def itinerary_as_table():
    """Convert the itinerary into a list of rows for the Gradio Dataframe."""
    return [[item["day"], item["activity"]] for item in sorted(itinerary, key=lambda x: x["day"])]




## Step 5 — Describing Tools to the LLM


In [18]:

#python
weather_function = {
    "name": "get_weather",
    "description": "Get the real current weather for a city. Use whenever a destination is mentioned.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "Name of the city"},
        },
        "required": ["city"],
    },
}

currency_function = {
    "name": "convert_currency",
    "description": "Convert an amount between currencies using live exchange rates.",
    "parameters": {
        "type": "object",
        "properties": {
            "amount": {"type": "number"},
            "from_currency": {"type": "string", "description": "3-letter code, e.g. USD"},
            "to_currency": {"type": "string", "description": "3-letter code, e.g. TRY"},
        },
        "required": ["amount", "from_currency", "to_currency"],
    },
}

itinerary_function = {
    "name": "add_to_itinerary",
    "description": "Save an agreed activity to the trip itinerary.",
    "parameters": {
        "type": "object",
        "properties": {
            "day": {"type": "integer", "description": "Day number of the trip, starting at 1"},
            "activity": {"type": "string", "description": "Short description of the activity"},
        },
        "required": ["day", "activity"],
    },
}

tools = [
    {"type": "function", "function": weather_function},
    {"type": "function", "function": currency_function},
    {"type": "function", "function": itinerary_function},
]

# Map tool names → real Python functions, so we can call them dynamically
TOOL_FUNCTIONS = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
    "add_to_itinerary": add_to_itinerary,
}

## Step 6 — Image Generation + Voice


In [28]:

def artist(city):
    image_response = openai.images.generate(
        model="gpt-image-1",
        prompt=f"A vibrant pop-art style postcard of a vacation in {city}, showing its most iconic sights",
        size="1024x1024",
        quality="low",        
        n=1,
        
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)     
    return Image.open(BytesIO(image_data))          

def talker(text):
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",      # fall back to "tts-1" if the account lacks access
        voice="onyx",
        input=text,
    )
    output_path = "agent_reply.mp3"
    with open(output_path, "wb") as f:
        f.write(response.content)
    return output_path                 # Gradio's Audio component happily accepts a filepath



🔧 Tool called: add_to_itinerary({'day': 1, 'activity': 'Visit the Blue Mosque, Hagia Sophia, Basilica Cistern, and Gülhane Park in Sultanahmet district'})
🔧 Tool called: add_to_itinerary({'day': 2, 'activity': 'Take a Bosphorus cruise, visit Topkapi Palace, and shop at the Grand Bazaar'})
🔧 Tool called: add_to_itinerary({'day': 3, 'activity': 'Explore Beyoğlu: walk Istiklal Street, visit Galata Tower, enjoy Turkish coffee, and visit Istanbul Modern Art Museum if time permits'})


C:\Users\ataka\AppData\Local\Temp\ipykernel_1244\3418252602.py:5: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 15, "Your Trip Itinerary", ln=True, align="C")
C:\Users\ataka\AppData\Local\Temp\ipykernel_1244\3418252602.py:8: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, f"Day {item['day']}:  {item['activity']}", ln=True)
Traceback (most recent call last):
  File "C:\Users\ataka\projects\llm_engineering\.venv\Lib\site-packages\fpdf\fpdf.py", line 5681, in normalize_text
    return text.encode(self.core_fonts_encoding).decode("latin-1")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'latin-1' codec can't encode character '\u011f' in position 20: ordinal not in range(256)

The above exception was the direct cause of the following exception:

Traceback (most recent call last):


## Step 7 — PDF Export


In [21]:
def create_pdf_itinerary():
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 20)
    pdf.cell(0, 15, "Your Trip Itinerary", ln=True, align="C")
    pdf.set_font("Helvetica", size=12)
    for item in sorted(itinerary, key=lambda x: x["day"]):
        pdf.cell(0, 10, f"Day {item['day']}:  {item['activity']}", ln=True)
    path = "itinerary.pdf"
    pdf.output(path)
    return path

## Step 8 — The Chat Brain (tool-calling loop)


In [25]:
last_city = None                                          

def chat(history):
    global last_city                                      
    messages = [{"role": "system", "content": system_message}] + history
    last_user_message = history[-1]["content"]

    if last_user_message.strip().lower() == "export":
        pdf_path = create_pdf_itinerary()
        history.append({"role": "assistant", "content": "Done! Your itinerary PDF is ready below. 🧳"})
        return history, None, None, itinerary_as_table(), pdf_path

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    image = None
    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        messages.append(assistant_msg)

        for tool_call in assistant_msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f"🔧 Tool called: {name}({args})")

            result = TOOL_FUNCTIONS[name](**args)

            if name == "get_weather" and "error" not in result:
                if result["city"] != last_city:           # ← CHANGED: only paint new cities
                    image = artist(result["city"])
                    last_city = result["city"]

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    voice_path = talker(reply)

    return history, voice_path, image, itinerary_as_table(), None

## Step 9 — The UI 


In [24]:

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

with gr.Blocks(title="TravelBuddy ✈️") as ui:
    gr.Markdown("# ✈️ TravelBuddy — your AI travel agent\nType **export** anytime to download your itinerary as a PDF.")
    with gr.Row():
        chatbot = gr.Chatbot(height=450, type="messages")
        image_output = gr.Image(height=450, interactive=False, label="Destination")
    with gr.Row():
        itinerary_table = gr.Dataframe(headers=["Day", "Activity"], label="📋 Your Itinerary", interactive=False)
        with gr.Column():
            audio_output = gr.Audio(autoplay=True, label="Agent voice")
            pdf_output = gr.File(label="📄 Itinerary PDF")
    with gr.Row():
        message = gr.Textbox(label="Chat with TravelBuddy:")

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output, itinerary_table, pdf_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


🔧 Tool called: get_weather({'city': 'Istanbul'})
